# Finetuning Qwen2.5 on Airflow DAGs (Local Mac Apple Silicon)

This notebook demonstrates how to fine-tune the **Qwen/Qwen2.5-Coder-1.5B-Instruct** model on a dataset of **Airflow DAGs** locally on **Apple Silicon** (M1/M2/M3) using the MPS backend.

Since `bitsandbytes` 4-bit/8-bit quantization is not reliably supported on MPS, this notebook loads the model in **float16** without quantization. The 1.5B model in fp16 requires ~3GB of memory, which fits comfortably in Apple Silicon's unified memory.

### Please note
**1. Training on Apple Silicon is significantly slower than on an A100 GPU.** This notebook is best suited for debugging, experimentation, and small-scale runs. For full training, use one of the Colab notebooks with an A100.

**2. If you want to fine-tune the model with your own settings**, you will need to modify:
- **`NEW_MODEL_NAME`** in the configuration cell — change the username to your own Hugging Face account (e.g., `your-username/qwen2.5-1.5b-airflow-instruct`)
- **Hugging Face Token** — ensure you have write permissions to push models to your account

## 1. Setup & Installation
We install the necessary libraries for fine-tuning with LoRA on Apple Silicon.

In [ ]:
%%capture
import os
import torch

!pip install transformers accelerate peft trl datasets huggingface_hub

In [ ]:
# Verify MPS availability
print(f"PyTorch version: {torch.__version__}")
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print(f"\u2705 MPS (Apple Silicon) backend available.")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"\u2705 CUDA GPU detected: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("\u26a0\ufe0f No GPU detected. Training will be very slow on CPU.")

print(f"Using device: {device}")

## 2. Configuration & Authentication
Log in to Hugging Face to access datasets and push your model.

In [ ]:
from huggingface_hub import login

print("Please provide your Hugging Face Token (Permissions: Write)")
login(add_to_git_credential=True)

In [ ]:
# Project Configuration
BASE_MODEL_NAME = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
DATASET_NAME = "andrea-t94/airflow-dag-dataset"

# Output Model Name (Change username if needed)
NEW_MODEL_NAME = "andrea-t94/qwen2.5-1.5b-airflow-instruct"

# Training Parameters
MAX_SEQ_LENGTH = 4096 # Fits most DAG files

## 3. Load Model with LoRA (no quantization)
We load the model in float16 without quantization (bitsandbytes is not supported on MPS). The 1.5B param model in fp16 uses ~3GB — well within Apple Silicon's unified memory.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
tokenizer.padding_side = "right"

# Load model in float16 (no quantization)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    torch_dtype=torch.float16,
    device_map={"":device},
    attn_implementation="sdpa",
)

# Enable gradient checkpointing to save memory
model.gradient_checkpointing_enable(gradient_checkpoint_kwargs={"use_reentrant": False})
model.enable_input_require_grads()

print(f"\u2705 Model loaded on {device} in float16")

# LoRA config (same as the other notebooks)
lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 4. Load & Format Dataset
We specificy a formatting function to apply the ChatML template (which Qwen uses) to our dataset.

In [ ]:
from datasets import load_dataset

# Load dataset
dataset = load_dataset(DATASET_NAME)

# Inspect dataset sizes
print(f"Train size: {len(dataset['train'])}")
if 'eval' in dataset: print(f"Eval size:  {len(dataset['eval'])}")

# Format function for ChatML
# The dataset should have a 'messages' column matching standard chat format
def formatting_prompts_func(examples):
    texts = []
    for messages in examples["messages"]:
        # Apply chat template but do NOT tokenize yet
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
        texts.append(text)
    return {"text": texts}

# Apply formatting
dataset = dataset.map(formatting_prompts_func, batched=True)

## 5. Training
Configure the `SFTTrainer`. We use a smaller batch size and more gradient accumulation steps to fit in memory. The `adamw_torch` optimizer is used instead of `adamw_8bit` (which requires bitsandbytes).

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset["train"],
    eval_dataset = dataset.get("eval"),
    dataset_text_field = "text",
    max_seq_length = MAX_SEQ_LENGTH,
    dataset_num_proc = 2,
    packing = True, # Set to True to speed up training if sequence len is variable and usually are shorter than max_seq_len
    args = TrainingArguments(
        per_device_train_batch_size = 1,  # Smaller batch for limited memory
        gradient_accumulation_steps = 32,  # effective_batch = 1*32 = 32 (same as other notebooks)
        warmup_steps = 10,
        max_steps = -1,                   # Set to -1 for full epochs
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = True,
        bf16 = False,
        logging_steps = 1,
        optim = "adamw_torch",            # Standard optimizer (no bitsandbytes needed)
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
        save_strategy = "steps",
        eval_strategy = "steps",
        eval_steps = 100,
        gradient_checkpointing = True,
        gradient_checkpointing_kwargs = {"use_reentrant": False},
    ),
)

In [ ]:
# Start Training
trainer_stats = trainer.train()

## 6. Save & Push to Hub
We save LoRA adapters and the full merged model, then push to Hugging Face Hub.

In [ ]:
# 1. Save LoRA Adapters only (Small file size, fast)
model.save_pretrained("lora_adapters")
tokenizer.save_pretrained("lora_adapters")
model.push_to_hub(f"{NEW_MODEL_NAME}-lora", token=True)
tokenizer.push_to_hub(f"{NEW_MODEL_NAME}-lora", token=True)

In [ ]:
# 2. Save Merged Model (Full model for direct inference)
from peft import AutoPeftModelForCausalLM

print("Merging LoRA weights into base model...")
merged_model = model.merge_and_unload()

# Save locally
merged_model.save_pretrained("merged_model", safe_serialization=True)
tokenizer.save_pretrained("merged_model")

# Push to Hub
print("Pushing merged model to Hub...")
merged_model.push_to_hub(NEW_MODEL_NAME, token=True, safe_serialization=True)
tokenizer.push_to_hub(NEW_MODEL_NAME, token=True)
print(f"Saved merged model to https://huggingface.co/{NEW_MODEL_NAME}")

In [ ]:
# 3. Convert to GGUF (for Ollama/Llama.cpp)
print("Converting to GGUF...")

# Install llama.cpp
!git clone https://github.com/ggerganov/llama.cpp.git /tmp/llama_cpp 2>/dev/null || true
!cd /tmp/llama_cpp && pip install -r requirements.txt 2>/dev/null

# Convert to GGUF f16 first
!python /tmp/llama_cpp/convert_hf_to_gguf.py merged_model --outfile merged_model.f16.gguf --outtype f16

# Quantize to Q4_K_M
!cd /tmp/llama_cpp && make -j quantize 2>/dev/null
!/tmp/llama_cpp/llama-quantize merged_model.f16.gguf merged_model.Q4_K_M.gguf Q4_K_M

# Upload GGUF to Hub
from huggingface_hub import HfApi
api = HfApi()
api.upload_file(
    path_or_fileobj="merged_model.Q4_K_M.gguf",
    path_in_repo="qwen2.5-coder-1.5b-instruct.Q4_K_M.gguf",
    repo_id=NEW_MODEL_NAME,
    token=True
)
print(f"GGUF uploaded to https://huggingface.co/{NEW_MODEL_NAME}")